In [0]:
#Step 1: Create Streaming Source

from pyspark.sql.functions import *
from pyspark.sql.types import *

df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 3) \
    .load()

In [0]:
#Step 2: Create Dummy Orders (simulate all cases)

orders = df.withColumn("order_id", col("value")) \
    .withColumn("amount", (rand()*100).cast("int")) \
    .withColumn(
        "event_time",
        expr("""
        CASE 
            WHEN value % 4 = 0 THEN timestamp - interval 1 minutes   -- ✅ on-time
            WHEN value % 4 = 1 THEN timestamp - interval 4 minutes   -- ✅ slightly late (accepted)
            WHEN value % 4 = 2 THEN timestamp - interval 8 minutes   -- ⚠️ borderline late
            ELSE timestamp - interval 15 minutes                     -- ❌ too late (will drop)
        END
        """)
    ) \
    .withColumn("arrival_time", col("timestamp"))


In [0]:
# Step 3: Apply Watermark
orders_wm = orders.withWatermark("event_time", "5 minutes")

In [0]:
# Step 4: Window Aggregation
result = orders_wm.groupBy(
    window("event_time", "10 minutes")
).agg(
    count("*").alias("total_orders"),
    sum("amount").alias("total_amount")
)

In [0]:
# Step 5: Output to Console
query = result.writeStream \
    .format("console") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("truncate", "false") \
    .option("checkpointLocation", "s3://customers-orders-bucket-v1/schema/order") \
    .start()

query.awaitTermination()

In [0]:
debug = orders.select(
    "order_id",
    "event_time",
    "arrival_time"
)

debug_query = debug.writeStream \
    .format("console") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("truncate", "false") \
    .option("checkpointLocation", "s3://customers-orders-bucket-v1/schema/order") \
    .start()